In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

2025-06-25 11:56:05.452765: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-25 11:56:05.529570: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Input shape: (batch_size, 13, 32)
input_seq = tf.keras.layers.Input(shape=(13, 32), name='input_sequence')

# Encoder LSTM
encoder_output, state_h, state_c = tf.keras.layers.LSTM(64, return_state=True, name='encoder_lstm')(input_seq)
encoder_states = [state_h, state_c]

# Repeat context vector for each of the 7 future time points
decoder_input = tf.keras.layers.RepeatVector(7)(encoder_output)

# Decoder LSTM
decoder_lstm = tf.keras.layers.LSTM(64, return_sequences=True, name='decoder_lstm')
decoder_output = decoder_lstm(decoder_input, initial_state=encoder_states)

# Output dense layer to produce 1 value per time step
output = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1), name='output')(decoder_output)

# Define and compile model
model = tf.keras.Model(inputs=input_seq, outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()

2025-06-25 11:56:06.906718: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-25 11:56:07.541086: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 281 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:06:00.0, compute capability: 7.0


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_sequence (InputLayer)    [(None, 13, 32)]     0           []                               
                                                                                                  
 encoder_lstm (LSTM)            [(None, 64),         24832       ['input_sequence[0][0]']         
                                 (None, 64),                                                      
                                 (None, 64)]                                                      
                                                                                                  
 repeat_vector (RepeatVector)   (None, 7, 64)        0           ['encoder_lstm[0][0]']           
                                                                                              

In [3]:
inputs = tf.keras.Input(shape=(13, 32))
x = tf.keras.layers.LSTM(16)(inputs)
outputs = tf.keras.layers.Dense(7)(x)
ts_similar_model = tf.keras.Model(inputs, outputs)

In [4]:
ts_similar_model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 13, 32)]          0         
                                                                 
 lstm (LSTM)                 (None, 16)                3136      
                                                                 
 dense_1 (Dense)             (None, 7)                 119       
                                                                 
Total params: 3,255
Trainable params: 3,255
Non-trainable params: 0
_________________________________________________________________
